In [16]:
import json

gt_path = "../results/rq1-6/gt.json"

pred_path = "../results/rq1-6/blagent_function_level_loc_gpt-oss.json"

In [17]:
with open(gt_path) as f:
    gt_data = json.load(f)

with open(pred_path) as f:
    pred_data_all = json.load(f)

In [18]:
pred_data_all[0]

{'swe_data_index': 0,
 'problem_statement': "Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels\nConsider the following model:\r\n\r\n```python\r\nfrom astropy.modeling import models as m\r\nfrom astropy.modeling.separable import separability_matrix\r\n\r\ncm = m.Linear1D(10) & m.Linear1D(5)\r\n```\r\n\r\nIt's separability matrix as you might expect is a diagonal:\r\n\r\n```python\r\n>>> separability_matrix(cm)\r\narray([[ True, False],\r\n       [False,  True]])\r\n```\r\n\r\nIf I make the model more complex:\r\n```python\r\n>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))\r\narray([[ True,  True, False, False],\r\n       [ True,  True, False, False],\r\n       [False, False,  True, False],\r\n       [False, False, False,  True]])\r\n```\r\n\r\nThe output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.\r\n\r\nIf however, I nest these compound 

In [19]:
locagent_preds = "../results/rq1-1/locagent/locagent_results_claude35sonnet/loc_outputs.jsonl"

with open(locagent_preds) as f:
    locagent_data = [json.loads(line) for line in f]

In [20]:
def normalize_locagent_output(locagent_entry):
    predicted_methods = locagent_entry.get("found_entities", [])

    formatted_entities = {}
    for method in predicted_methods:
        path, method_name = method.split(":")
        path = path.strip()
        method_name = method_name.replace(".", "::").strip()
        
        # Append the method name to the list of methods for the corresponding file path
        if path not in formatted_entities:
            formatted_entities[path] = []
        formatted_entities[path].append(method_name)
    
    formatted_list = []
    for path, methods in formatted_entities.items():
        formatted_list.append({path: methods})
    
    return formatted_list

In [21]:
normalized_locagent_data = []

for entry in locagent_data:
    normalized_entry = normalize_locagent_output(entry)
    normalized_locagent_data.append({
        "instance_id": entry["instance_id"],
        "predicted_methods": normalized_entry
    })

In [22]:
len(normalized_locagent_data)

300

In [23]:
# Filtered instances are based on LocAgent
# https://github.com/gersteinlab/LocAgent/blob/master/evaluation/eval_metric.py

filtered_instances=['pytest-dev__pytest-5227',
 'sympy__sympy-15345',
 'sympy__sympy-21614',
 'scikit-learn__scikit-learn-13439',
 'sympy__sympy-11400',
 'sympy__sympy-19487',
 'sympy__sympy-15308',
 'django__django-12915',
 'sympy__sympy-20590',
 'sympy__sympy-17022',
 'django__django-11099',
 'django__django-13220',
 'django__django-11964',
 'matplotlib__matplotlib-25332',
 'django__django-10914',
 'django__django-14915',
 'django__django-11049',
 'django__django-11564',
 'sympy__sympy-17655',
 'sympy__sympy-16106',
 'sympy__sympy-12171',
 'django__django-15400',
 'django__django-14411',
 'sympy__sympy-21055',
 'django__django-15213',
 'django__django-15902',
 ]

In [24]:
gt_list = []
pred_data = []

# Keep only those entries that have a function level ground truth (i.e., those that contain "::" in the first element of the value list)
for i, (k, v) in enumerate(gt_data.items()):
    if k in filtered_instances:
        continue
    # if "::" in v[0]:
    gt_list.append(v)
    
    pred_data.append(pred_data_all[i])

    # norm_loc_out = [entry for entry in normalized_locagent_data if entry["instance_id"] == k][0]
    # pred_data.append({
    #     "instance_id": k,
    #     "final_reranked_files": norm_loc_out["predicted_methods"]
    # })


In [25]:
gt_list[0]

['astropy/modeling/separable.py::_cstack']

In [26]:
# Debug: Inspect data format
print("=== DATA FORMAT INSPECTION ===\n")

print("Sample GT (first 3):")
for i, gt in enumerate(gt_list[:3]):
    print(f"{i}: {gt}")

print("\n\nSample Prediction (first item):")
print(f"Keys: {pred_data[0].keys()}")
print(f"\nFinal reranked files (first 3):")
for i, f in enumerate(pred_data[0]["final_reranked_files"][:3]):
    print(f"{i}: {f}")
    
print(f"\n\nTotal predictions: {len(pred_data)}")
print(f"Total GT: {len(gt_list)}")

=== DATA FORMAT INSPECTION ===

Sample GT (first 3):
0: ['astropy/modeling/separable.py::_cstack']
1: ['astropy/io/ascii/rst.py::get_fixedwidth_params', 'astropy/io/ascii/rst.py::RST.__init__', 'astropy/io/ascii/rst.py::RST.write']
2: ['astropy/io/ascii/qdp.py::_line_type', 'astropy/io/ascii/qdp.py::_get_tables_from_qdp_file']


Sample Prediction (first item):
Keys: dict_keys(['swe_data_index', 'problem_statement', 'augmented_query', 'patch_file', 'is_patch_in_top10_t0', 'is_patch_in_top10_t1', 'retrieved_files_t0', 'retrieved_files_t1', 'ranked_scores', 'ranked_files', 'final_reranked_files', 'token_usage'])

Final reranked files (first 3):
0: {'astropy/modeling/separable.py': ['_separable', '_cstack', 'separability_matrix']}
1: {'astropy/modeling/core.py': ['Model::_calculate_separability_matrix', 'CompoundModel::__init__', 'CompoundModel::evaluate']}
2: {'astropy/modeling/mappings.py': ['Mapping::evaluate']}


Total predictions: 274
Total GT: 274


In [27]:
# Calculate Top-K Accuracy (Simplified)
def calculate_top_k_accuracy(pred_data, gt_list, k=5):
    """
    Simple top-k accuracy: check if any predicted method appears in any GT string.
    """
    correct = 0
    total = len(pred_data)
    results = []
    
    for i, p in enumerate(pred_data):
        ranked_files = p.get("final_reranked_files", [])
        
        # Collect top-k predicted methods
        pred_methods = []
        for file_dict in ranked_files:
            methods = list(file_dict.values())[0]
            pred_methods.extend(methods)
            
            if len(pred_methods) >= k:
                break
        
        # Only use top-k
        pred_methods = pred_methods[:k]
        
        # Get ground truth (list of strings like "file.py::method_name")
        gt_strings = gt_list[i] if i < len(gt_list) else []
        
        # Check if any predicted method appears in any GT string
        found = False
        matched_pred = None
        matched_gt = None
        
        for pred_method in pred_methods:
            # Normalize prediction: replace :: with . for comparison
            pred_normalized = pred_method.replace("::", ".")
            
            for gt_string in gt_strings:
                # Split into file and method parts
                if '::' in gt_string:
                    method_part = gt_string.split('::')[1]
                else:
                    method_part = gt_string  # file-only GT, fallback

                if pred_method in method_part or pred_normalized in method_part:
                    found = True
                    matched_pred = pred_method
                    matched_gt = gt_string
                    break
            if found:
                break
        
        if found:
            correct += 1
        
        results.append({
            "index": i,
            "found": found,
            "gt": gt_strings,
            "predictions": pred_methods,
            "matched_pred": matched_pred,
            "matched_gt": matched_gt
        })
    
    accuracy = correct / total if total > 0 else 0
    return accuracy, correct, total, results

# Calculate Top-5 Accuracy
print("Calculating Top-10 Accuracy...")
accuracy, correct, total, results = calculate_top_k_accuracy(pred_data, gt_list, k=10)

print(f"\n{'='*60}")
print(f"Top-10 Accuracy: {accuracy:.4f} ({correct}/{total})")
print(f"{'='*60}\n")

# Show first few results
print("Sample Results:")
for i, res in enumerate(results[120:130]):
    print(f"\nInstance {i}: {'✓ FOUND' if res['found'] else '✗ NOT FOUND'}")
    print(f"  GT: {res['gt']}")
    print(f"  Top-10: {res['predictions']}")
    if res['matched_pred']:
        print(f"  Match: '{res['matched_pred']}' in '{res['matched_gt']}'")

Calculating Top-10 Accuracy...

Top-10 Accuracy: 0.8102 (222/274)

Sample Results:

Instance 0: ✓ FOUND
  GT: ['lib/matplotlib/style/core.py', 'lib/matplotlib/style/core.py::fix_style', 'lib/matplotlib/style/core.py::reload_library']
  Top-10: ['use', 'reload_library', 'read_style_directory', '__getattr__::__version__']
  Match: 'reload_library' in 'lib/matplotlib/style/core.py::reload_library'

Instance 1: ✓ FOUND
  GT: ['lib/matplotlib/axis.py::Axis.set_ticks']
  Top-10: ['Axis::set_ticks', 'Axis::_apply_params', 'Axis::set_ticklabels', 'Axes::set_xticks', 'Axes::set_yticks', 'Axes::set_xticklabels', 'Axes::set_yticklabels', 'xticks', 'yticks', 'Axes3D::set_xticks']
  Match: 'Axis::set_ticks' in 'lib/matplotlib/axis.py::Axis.set_ticks'

Instance 2: ✓ FOUND
  GT: ['lib/matplotlib/colors.py::Colormap.__call__']
  Top-10: ['Colormap::__call__', 'Colormap::set_over', 'Colormap::set_under', 'ScalarMappable::to_rgba', 'ScalarMappable::set_cmap', 'ScalarMappable::set_norm', '_ImageBase::set

In [28]:
# Show failed cases
failed = [r for r in results if not r['found']]

print(f"\nFailed Cases: {len(failed)}/{total}")
print("\nFirst 5 failed cases:")
for i, case in enumerate(failed):
    print(f"\n--- Case {case['index']} ---")
    print(f"GT: {case['gt']}")
    print(f"Predictions: {case['predictions']}")


Failed Cases: 52/274

First 5 failed cases:

--- Case 6 ---
GT: ['django/db/models/fields/__init__.py::Field.formfield']
Predictions: ['FilePathField::__init__', 'FilePathField::deconstruct', 'FilePathField::formfield', 'ModelFieldSerializer::serialize', 'FunctionTypeSerializer::serialize', 'DeconstructableSerializer::serialize', 'MigrationWriter::serialize', 'OperationWriter::serialize', 'AddField::state_forwards', 'AlterField::state_forwards']

--- Case 7 ---
GT: ['django/db/models/sql/compiler.py::SQLCompiler.__init__']
Predictions: ['SQLCompiler::get_order_by', 'SQLCompiler::as_sql', 'SQLCompiler::compile', 'Query::add_ordering', 'Query::clear_ordering', 'get_order_dir', 'QuerySet::order_by', 'QuerySet::_earliest', 'QuerySet::ordered', 'RawSQL::as_sql']

--- Case 13 ---
GT: ['django/utils/autoreload.py::iter_modules_and_files']
Predictions: ['run_with_reloader', 'BaseReloader::watched_files', 'StatReloader::snapshot_files', 'Command::run', 'Command::inner_run', 'ManagementUtility:

In [29]:
# Compare different K values
print("\nTop-K Accuracy:")
print(f"{'K':<5} {'Accuracy':<12} {'Correct/Total'}")
print("-" * 40)

for k in [5, 10]:
    acc, corr, tot, _ = calculate_top_k_accuracy(pred_data, gt_list, k=k)
    print(f"Top-{k:<2}  {acc:.4f} ({acc*100:5.2f}%)  {corr}/{tot}")


Top-K Accuracy:
K     Accuracy     Correct/Total
----------------------------------------
Top-5   0.7810 (78.10%)  214/274
Top-10  0.8102 (81.02%)  222/274
